# EXAONE 3.5 7.8B QLoRA 파인튜닝 연습 — 문장에서 정보 추출 → JSON

**목표:** 한국어 문장에서 인물·장소·기관·날짜·시간·수량을 뽑아 고정된 JSON으로 출력하도록 학습하고, 튜닝 전후 성능을 숫자로 비교합니다.

- 모델: `LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct` (비상업·연구용 라이선스 — 연습용)
- 데이터: KLUE-NER (CC BY-SA 4.0) → 학습 1,000건 / 평가 200건
- 방식: QLoRA (원본 가중치 4비트로 고정 + LoRA 어댑터만 학습)

| 순서 | 셀 | 예상 시간 (무료 T4) |
|---|---|---|
| 1~3 | GPU 확인 · 드라이브 연결 · 설치 | 수 분 |
| 4 | 데이터 준비 | 1분 |
| 5 | 모델 불러오기 + 맛보기 | 10분 안팎 (첫 다운로드 약 15GB) |
| 6~7 | 튜닝 **전** 평가 200건 | 10~20분 |
| 8 | 학습 (1,000건 × 1 epoch) | 30분~1시간 20분 |
| 9~10 | 튜닝 **후** 평가 + 비교 | 10~20분 |

**끊김 대비:** 결과는 모두 구글 드라이브 `MyDrive/ft_practice`에 저장됩니다. 학습 중 끊기면 1~5번을 다시 실행한 뒤 8번을 실행하면 마지막 checkpoint부터 이어서 학습합니다. 학습이 끝난 뒤 끊겼다면 맨 아래 **"끊긴 뒤 재개"** 셀을 쓰세요.

> 시작 전에 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**인지 확인하세요.

## 1. GPU 확인
`Tesla T4`, 15GB 안팎이 보이면 정상입니다.

In [ ]:
!nvidia-smi

## 2. 구글 드라이브 연결 (끊김 대비 저장소)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = "/content/drive/MyDrive/ft_practice"
os.makedirs(WORK_DIR, exist_ok=True)
print("저장 위치:", WORK_DIR)

## 3. 라이브러리 설치
설치가 끝나면 **런타임 → 세션 다시 시작**을 한 뒤 **2번 셀부터** 다시 실행하세요.

`cudf-cu13 … requires pyarrow<24` 같은 dependency conflicts 경고는 무시해도 됩니다. 코랩에 미리 깔린 cudf(이 노트북에서 안 씀)와의 버전 충돌일 뿐입니다.

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets

## 4. 데이터 준비 (학습 1,000건 / 평가 200건)
KLUE-NER 문장은 `<경찰:OG>은 … <박:PS>씨의 딸(<32:QT>)…` 처럼 정답이 꺾쇠로 표시되어 있습니다. 꺾쇠를 벗겨 **입력 문장**을 만들고, 꺾쇠 안의 값을 모아 **정답 JSON**을 만듭니다.

- 평가 데이터는 학습용(train)이 아닌 검증용(validation) 묶음에서 뽑으므로 겹치지 않습니다.
- `prompt` / `completion`으로 나눠 두면 SFTTrainer가 **답(completion) 부분에만 손실**을 겁니다.

In [ ]:
import re, json, random
from datasets import load_dataset

raw = load_dataset("klue/klue", "ner")

PAT = re.compile(r"<([^<>]+?):(DT|LC|OG|PS|QT|TI)>")
LABELS = {"PS": "인물", "LC": "장소", "OG": "기관", "DT": "날짜", "TI": "시간", "QT": "수량"}
KEYS = list(LABELS.values())

SYSTEM = "당신은 문장에서 정보를 추출해 JSON으로만 답하는 도구입니다."
INSTRUCTION = (
    "다음 문장에서 인물, 장소, 기관, 날짜, 시간, 수량을 찾아 아래 형식의 JSON으로만 답하세요.\n"
    "해당하는 것이 없으면 빈 목록 []으로 두세요. 문장에 없는 내용은 만들지 마세요.\n"
    '형식: {"인물": [], "장소": [], "기관": [], "날짜": [], "시간": [], "수량": []}\n\n'
    "문장: "
)

def to_example(tagged):
    # 태그 문장 → 학습/평가 예시. 원문 자체에 꺾쇠가 있어 변환이 깨지면 None
    text = PAT.sub(lambda m: m.group(1), tagged)
    if "<" in text or ">" in text:
        return None
    answer = {k: [] for k in KEYS}
    for ent, tag in PAT.findall(tagged):
        if ent not in answer[LABELS[tag]]:
            answer[LABELS[tag]].append(ent)
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": INSTRUCTION + text},
        ],
        "completion": [
            {"role": "assistant", "content": json.dumps(answer, ensure_ascii=False)},
        ],
        "text": text,
        "answer": answer,
    }

def build(split, n, seed=42):
    rows = [ex for s in raw[split]["sentence"] if (ex := to_example(s))]
    random.Random(seed).shuffle(rows)
    return rows[:n]

train_rows = build("train", 1000)
test_rows = build("validation", 200)

# 학습 파일에는 prompt/completion만 남김 (text·answer는 평가 때만 사용)
with open(f"{WORK_DIR}/train.jsonl", "w", encoding="utf-8") as f:
    for r in train_rows:
        f.write(json.dumps({"prompt": r["prompt"], "completion": r["completion"]}, ensure_ascii=False) + "\n")
with open(f"{WORK_DIR}/test.jsonl", "w", encoding="utf-8") as f:
    for r in test_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("학습", len(train_rows), "/ 평가", len(test_rows))
print("입력 예:", test_rows[0]["text"])
print("정답 예:", test_rows[0]["completion"][0]["content"])

## 5. 모델 불러오기 (4비트) + 맛보기
- 처음 실행하면 원본 가중치 약 15GB를 받습니다. 4비트로 바꿔 올리면 GPU 메모리는 6GB 안팎을 씁니다.
- T4는 bf16을 지원하지 않아 계산은 fp16으로 합니다.
- **에러가 나면:** EXAONE 3.5는 자체 모델 코드(`trust_remote_code`)를 쓰므로 최신 transformers와 안 맞을 수 있습니다. 에러 메시지를 그대로 알려 주세요.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,   # T4는 bf16 미지원 → fp16
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", trust_remote_code=True
)
model.eval()

def generate_batch(prompts, max_new_tokens=200):
    # 여러 프롬프트를 한 번에 생성 (생성은 왼쪽 패딩이어야 함)
    tokenizer.padding_side = "left"
    texts = [tokenizer.apply_chat_template(p, add_generation_prompt=True, tokenize=False) for p in prompts]
    enc = tokenizer(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.pad_token_id)
    tokenizer.padding_side = "right"
    return tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"GPU 메모리 사용: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
for r, o in zip(test_rows[:3], generate_batch([r["prompt"] for r in test_rows[:3]])):
    print("문장:", r["text"])
    print("모델:", o.strip())
    print("정답:", r["completion"][0]["content"])
    print("-" * 70)

## 6. 평가 함수
정답 JSON과 비교해 코드로 채점합니다 (judge 불필요).

| 지표 | 뜻 |
|---|---|
| 순수JSON비율 | 출력 전체가 `{…}` 하나뿐인가 (앞뒤 설명 없음) |
| 파싱성공률 | 출력에서 JSON을 읽어낼 수 있는가 |
| 스키마준수율 | 6개 키가 정확히 있고 값이 모두 문자열 목록인가 |
| 정밀도 / 재현율 / F1 | (항목, 값) 쌍 기준. 파싱 실패는 전부 놓친 것으로 계산 |
| 원문에없는값비율 | 뽑은 값 중 입력 문장에 아예 없는 문자열 비율 — **지어낸 값(할루시네이션) 지표** |

생성 결과는 배치마다 드라이브에 저장되어, 중간에 끊겨도 다시 실행하면 이어서 진행합니다.

In [ ]:
def parse_output(s):
    start, end = s.find("{"), s.rfind("}")
    if start == -1 or end <= start:
        return None
    try:
        obj = json.loads(s[start:end + 1])
    except json.JSONDecodeError:
        return None
    return obj if isinstance(obj, dict) else None

def schema_ok(obj):
    return set(obj.keys()) == set(KEYS) and all(
        isinstance(v, list) and all(isinstance(x, str) for x in v) for v in obj.values()
    )

def prf(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return p, r, f

def run_generation(name, rows, batch_size=8):
    # 결과를 preds_{name}.jsonl에 배치마다 덧붙임. 이미 있는 idx는 건너뜀
    path = f"{WORK_DIR}/preds_{name}.jsonl"
    done = {}
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)
                done[d["idx"]] = d["output"]
    todo = [i for i in range(len(rows)) if i not in done]
    print(f"[{name}] 완료 {len(done)} / 남음 {len(todo)}")
    with open(path, "a", encoding="utf-8") as f:
        for b in range(0, len(todo), batch_size):
            idxs = todo[b:b + batch_size]
            outs = generate_batch([rows[i]["prompt"] for i in idxs])
            for i, o in zip(idxs, outs):
                done[i] = o
                f.write(json.dumps({"idx": i, "output": o}, ensure_ascii=False) + "\n")
            f.flush()
            print(f"  {len(done)}/{len(rows)}", end="\r")
    print()
    return [done[i] for i in range(len(rows))]

def score(name, rows, outputs):
    strict = parsed = schema = 0
    tp = fp = fn = 0
    per = {k: [0, 0, 0] for k in KEYS}
    pred_total = not_in_text = 0
    for r, out in zip(rows, outputs):
        s = out.strip()
        if s.startswith("{") and s.endswith("}"):
            strict += 1
        obj = parse_output(out)
        pred = {k: set() for k in KEYS}
        if obj is not None:
            parsed += 1
            schema += schema_ok(obj)
            for k in KEYS:
                v = obj.get(k)
                if isinstance(v, list):
                    pred[k] = {x for x in v if isinstance(x, str)}
        for k in KEYS:
            gold = set(r["answer"][k])
            t, p_, n_ = len(pred[k] & gold), len(pred[k] - gold), len(gold - pred[k])
            tp += t; fp += p_; fn += n_
            per[k][0] += t; per[k][1] += p_; per[k][2] += n_
            for x in pred[k]:
                pred_total += 1
                not_in_text += x not in r["text"]
    n = len(rows)
    p, rc, f = prf(tp, fp, fn)
    m = {
        "순수JSON비율": round(strict / n, 3),
        "파싱성공률": round(parsed / n, 3),
        "스키마준수율": round(schema / n, 3),
        "정밀도": round(p, 3),
        "재현율": round(rc, 3),
        "F1": round(f, 3),
        "원문에없는값비율": round(not_in_text / pred_total, 3) if pred_total else 0.0,
        "항목별_F1": {k: round(prf(*per[k])[2], 3) for k in KEYS},
    }
    with open(f"{WORK_DIR}/metrics_{name}.json", "w", encoding="utf-8") as fo:
        json.dump(m, fo, ensure_ascii=False, indent=2)
    print(json.dumps(m, ensure_ascii=False, indent=2))
    return m

## 7. 튜닝 **전** 평가 (원본 모델, 200건)

In [ ]:
base_outputs = run_generation("base", test_rows)
base_metrics = score("base", test_rows, base_outputs)

## 8. 학습 (QLoRA)
- `target_modules="all-linear"`: 모든 선형층에 LoRA를 붙입니다 (모듈 이름을 몰라도 됨).
- 실효 배치 16 (4 × 누적 4) → 1,000건 1 epoch ≈ 63 step. 25 step마다 드라이브에 checkpoint 저장.
- 끊긴 뒤 다시 실행하면 마지막 checkpoint부터 이어서 학습합니다 (1~5번 먼저 실행).
- 효과가 부족하면 `num_train_epochs=2`로 늘려 보세요 (시간 2배).

In [ ]:
import glob, time
from datasets import load_dataset as hf_load
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

train_ds = hf_load("json", data_files=f"{WORK_DIR}/train.jsonl", split="train")

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM", target_modules="all-linear",
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()   # 전체 중 학습되는 비율 확인

CKPT_DIR = f"{WORK_DIR}/checkpoints"
args = SFTConfig(
    output_dir=CKPT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=5,
    logging_steps=5,
    save_steps=25,
    save_total_limit=2,
    fp16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    max_length=512,
    report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=train_ds, processing_class=tokenizer)

resume = bool(glob.glob(f"{CKPT_DIR}/checkpoint-*"))
print("checkpoint에서 이어서 학습:", resume)
t0 = time.time()
trainer.train(resume_from_checkpoint=resume)
print(f"학습 시간: {(time.time() - t0) / 60:.1f}분")

ADAPTER_DIR = f"{WORK_DIR}/adapter"
trainer.save_model(ADAPTER_DIR)          # LoRA 어댑터만 저장 (수십 MB)
tokenizer.save_pretrained(ADAPTER_DIR)
print("어댑터 저장:", ADAPTER_DIR)

## 9. 튜닝 **후** 평가 (200건)
학습 로그의 `loss`가 내려갔는지도 위에서 확인해 보세요. loss 감소는 "학습이 돌아간다"는 확인일 뿐이고, 실제 효과는 아래 평가 숫자로 판단합니다.

In [ ]:
model = trainer.model
model.eval()
model.config.use_cache = True

ft_outputs = run_generation("ft", test_rows)
ft_metrics = score("ft", test_rows, ft_outputs)

## 10. 전후 비교

In [ ]:
base = json.load(open(f"{WORK_DIR}/metrics_base.json", encoding="utf-8"))
ft = json.load(open(f"{WORK_DIR}/metrics_ft.json", encoding="utf-8"))

print(f"{'지표':<14}{'튜닝 전':>10}{'튜닝 후':>10}")
for k, v in base.items():
    if k != "항목별_F1":
        print(f"{k:<14}{v:>10}{ft[k]:>10}")
print("\n항목별 F1")
for k in KEYS:
    print(f"  {k:<10}{base['항목별_F1'][k]:>10}{ft['항목별_F1'][k]:>10}")

def load_preds(name):
    with open(f"{WORK_DIR}/preds_{name}.jsonl", encoding="utf-8") as f:
        d = {x["idx"]: x["output"] for x in map(json.loads, f)}
    return [d[i] for i in range(len(test_rows))]

# 튜닝 전후 출력이 달라진 예시 3개
bo, fo = load_preds("base"), load_preds("ft")
shown = 0
for r, b, f_ in zip(test_rows, bo, fo):
    if b.strip() != f_.strip():
        print("\n문장:", r["text"])
        print("전  :", b.strip()[:300])
        print("후  :", f_.strip()[:300])
        print("정답:", r["completion"][0]["content"])
        shown += 1
        if shown == 3:
            break

---
## (선택) 끊긴 뒤 재개 — 학습이 끝난 어댑터 불러오기
학습(8번)이 끝나 어댑터가 드라이브에 저장된 뒤 세션이 끊겼다면:
1. 1~5번 셀을 다시 실행 (원본 모델 불러오기)
2. 아래 셀 실행 → 어댑터를 원본 모델에 얹음
3. 9번 셀에서 `model = trainer.model` 줄만 지우고 실행 → 10번 실행

**어댑터가 실제로 적용됐는지 점검:** 아래 셀은 어댑터를 얹기 전과 후의 출력을 비교합니다. 두 출력이 완전히 같다면 적용이 안 된 것입니다.

In [ ]:
from peft import PeftModel

sample = [test_rows[0]["prompt"]]
before = generate_batch(sample)[0]

model = PeftModel.from_pretrained(model, f"{WORK_DIR}/adapter")
model.eval()
after = generate_batch(sample)[0]

print("어댑터 전:", before.strip())
print("어댑터 후:", after.strip())
print("적용됨" if before.strip() != after.strip() else "⚠ 출력이 같음 — 어댑터 적용 확인 필요")